In [26]:
import warnings
warnings.filterwarnings("ignore")
from yfinance import download
from pandas import DateOffset, Timestamp

In [3]:
df = download('^bvsp', period='20y', interval='1d' ,auto_adjust=False, progress=False).droplevel(1, axis=1)
df.tail(3)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2025-12-11,159189.000000,159189.000000,159850.000000,158098.000,159072.000,7018000
2025-12-12,160766.000000,160766.000000,161263.000000,159189.000,159189.000,7671200
2025-12-15,162481.734375,162481.734375,163073.140625,160766.375,160766.375,0


In [4]:
def media_acumulada(df_values):
    lista = []
    value = 0
    for i, v in enumerate(df_values):
        value += v
        lista.append(value / i)
    return lista

In [5]:
df['retorno'] = df['Adj Close'].pct_change(1)
df['sd'] = df['retorno'].rolling(60).std()

df = df.dropna()

df['sinal_0'] = media_acumulada(df['sd'].values)
df['sinal_1'] = df['sinal_0'] * 2  

df = df.dropna()



df = df.iloc[1:]

df

Price,Adj Close,Close,High,Low,Open,Volume,retorno,sd,sinal_0,sinal_1
Date,,,,,,,,,,
2006-03-16,38157.000000,38157.000000,38578.000000,37962.000,38245.000,0,-0.002275,0.013665,0.027316,0.054633
2006-03-17,38049.000000,38049.000000,38268.000000,37769.000,38159.000,0,-0.002830,0.013606,0.020461,0.040923
2006-03-20,38204.000000,38204.000000,38500.000000,37978.000,38049.000,0,0.004074,0.013608,0.018177,0.036353
2006-03-21,37398.000000,37398.000000,38204.000000,37351.000,38204.000,0,-0.021097,0.013863,0.017098,0.034196
2006-03-22,37851.000000,37851.000000,37851.000000,37156.000,37398.000,0,0.012113,0.013922,0.016463,0.032926
...,...,...,...,...,...,...,...,...,...,...
2025-12-09,157981.000000,157981.000000,158851.000000,155188.000,158187.000,8703800,-0.001302,0.008943,0.014884,0.029768
2025-12-10,159075.000000,159075.000000,159691.000000,157628.000,157984.000,8239900,0.006925,0.008966,0.014883,0.029765
2025-12-11,159189.000000,159189.000000,159850.000000,158098.000,159072.000,7018000,0.000717,0.008889,0.014881,0.029763


In [6]:
import plotly.express as px
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=(
        "Preço de Fechamento",
        "Retornos",
        "Desvio Padrão"
    )
)

# --- 1. Preço de Fechamento ---
fig.add_scatter(
    x=df.index,
    y=df["Close"],
    mode="lines",
    name="Close",
    row=1,
    col=1
)

# --- 2. Retornos ---
fig.add_scatter(
    x=df.index,
    y=df["retorno"],
    mode="lines",
    name="Retorno",
    row=2,
    col=1
)

# Linha zero (referência)
fig.add_hline(
    y=0,
    line_width=1,
    line_dash="dash",
    row=2,
    col=1
)

# --- 3. Volatilidade + Sinais ---
fig.add_scatter(
    x=df.index,
    y=df["sd"],
    mode="lines",
    name="sd",
    row=3,
    col=1
)

fig.add_scatter(
    x=df.index,
    y=df["sinal_0"],
    mode="lines",
    name="Sinal 0",
    row=3,
    col=1
)

fig.add_scatter(
    x=df.index,
    y=df["sinal_1"],
    mode="lines",
    name="Sinal 1",
    row=3,
    col=1
)

# --- Layout profissional ---
fig.update_layout(
    height=800,
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Séries",
    margin=dict(t=70, b=40, l=60, r=40)
)

fig.update_xaxes(title_text=None)
fig.update_yaxes(title_text="Preço", row=1, col=1)
fig.update_yaxes(title_text="Retorno", row=2, col=1)
fig.update_yaxes(title_text="sd", row=3, col=1)

fig.show()


In [7]:
df

Price,Adj Close,Close,High,Low,Open,Volume,retorno,sd,sinal_0,sinal_1
Date,,,,,,,,,,
2006-03-16,38157.000000,38157.000000,38578.000000,37962.000,38245.000,0,-0.002275,0.013665,0.027316,0.054633
2006-03-17,38049.000000,38049.000000,38268.000000,37769.000,38159.000,0,-0.002830,0.013606,0.020461,0.040923
2006-03-20,38204.000000,38204.000000,38500.000000,37978.000,38049.000,0,0.004074,0.013608,0.018177,0.036353
2006-03-21,37398.000000,37398.000000,38204.000000,37351.000,38204.000,0,-0.021097,0.013863,0.017098,0.034196
2006-03-22,37851.000000,37851.000000,37851.000000,37156.000,37398.000,0,0.012113,0.013922,0.016463,0.032926
...,...,...,...,...,...,...,...,...,...,...
2025-12-09,157981.000000,157981.000000,158851.000000,155188.000,158187.000,8703800,-0.001302,0.008943,0.014884,0.029768
2025-12-10,159075.000000,159075.000000,159691.000000,157628.000,157984.000,8239900,0.006925,0.008966,0.014883,0.029765
2025-12-11,159189.000000,159189.000000,159850.000000,158098.000,159072.000,7018000,0.000717,0.008889,0.014881,0.029763


In [8]:
lista_evento_0 = []
lista_evento_1 = []

for i in range(len(df)):

    data = df.index
    desvio_padrao = df['sd']
    sinal_0 = df['sinal_0']
    sinal_1 = df['sinal_1']

    # sinal 0
    if desvio_padrao.iloc[i-1] <= sinal_0.iloc[i-1] and desvio_padrao.iloc[i] >= sinal_0.iloc[i]:
        lista_evento_0.append(True)
    else:
        lista_evento_0.append(False)

    # sinal 1
    if desvio_padrao.iloc[i-1] <= sinal_1.iloc[i-1] and desvio_padrao.iloc[i] >= sinal_1.iloc[i]:
        lista_evento_1.append(True)
    else:
        lista_evento_1.append(False)

df['evento_0'] = lista_evento_0
df['evento_1'] = lista_evento_1

df


Price,Adj Close,Close,High,Low,Open,Volume,retorno,sd,sinal_0,sinal_1,evento_0,evento_1
Date,,,,,,,,,,,,
2006-03-16,38157.000000,38157.000000,38578.000000,37962.000,38245.000,0,-0.002275,0.013665,0.027316,0.054633,False,False
2006-03-17,38049.000000,38049.000000,38268.000000,37769.000,38159.000,0,-0.002830,0.013606,0.020461,0.040923,False,False
2006-03-20,38204.000000,38204.000000,38500.000000,37978.000,38049.000,0,0.004074,0.013608,0.018177,0.036353,False,False
2006-03-21,37398.000000,37398.000000,38204.000000,37351.000,38204.000,0,-0.021097,0.013863,0.017098,0.034196,False,False
2006-03-22,37851.000000,37851.000000,37851.000000,37156.000,37398.000,0,0.012113,0.013922,0.016463,0.032926,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-09,157981.000000,157981.000000,158851.000000,155188.000,158187.000,8703800,-0.001302,0.008943,0.014884,0.029768,False,False
2025-12-10,159075.000000,159075.000000,159691.000000,157628.000,157984.000,8239900,0.006925,0.008966,0.014883,0.029765,False,False
2025-12-11,159189.000000,159189.000000,159850.000000,158098.000,159072.000,7018000,0.000717,0.008889,0.014881,0.029763,False,False


In [9]:
df_evento_0 = df[df['evento_0'] == True]
df_evento_1 = df[df['evento_1'] == True]

In [10]:
fig = px.line(
    df,
    x=df.index,
    y=[
        "sd", "sinal_0", "sinal_1"
        ],
    title="Desvio Padrão com Cruzamentos do Sinal 0 e 1",
    labels={
        "value": "Valor",
        "variable": "Série",
        "x": "Data"
    }
)

fig.add_scatter(
    x=df_evento_0.index,
    y=df_evento_0["sd"],
    mode="markers",
    name="Cruzamento sinal_0",
    marker=dict(size=9)
)

fig.add_scatter(
    x=df_evento_1.index,
    y=df_evento_1["sd"],
    mode="markers",
    name="Cruzamento sinal_1",
    marker=dict(size=9)
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    height=500
)

fig.show()


In [27]:
# intervalos de tempo que finaliza no momento de alta vol do ibov

# sinal 0
for data_alta_vol in df[df['evento_0'] == True].index:
    print("start =", str(Timestamp(data_alta_vol) - DateOffset(years=10))[:10], "| end =", str(data_alta_vol)[:10])

start = 1996-05-22 | end = 2006-05-22
start = 1996-09-01 | end = 2006-09-01
start = 1997-02-27 | end = 2007-02-27
start = 1997-08-09 | end = 2007-08-09
start = 1998-06-25 | end = 2008-06-25
start = 1998-07-30 | end = 2008-07-30
start = 1999-06-15 | end = 2009-06-15
start = 2001-08-11 | end = 2011-08-11
start = 2002-07-27 | end = 2012-07-27
start = 2004-10-13 | end = 2014-10-13
start = 2005-10-13 | end = 2015-10-13
start = 2005-10-30 | end = 2015-10-30
start = 2005-12-08 | end = 2015-12-08
start = 2005-12-18 | end = 2015-12-18
start = 2006-01-04 | end = 2016-01-04
start = 2006-01-29 | end = 2016-01-29
start = 2006-06-21 | end = 2016-06-21
start = 2006-06-23 | end = 2016-06-23
start = 2007-05-19 | end = 2017-05-19
start = 2007-06-06 | end = 2017-06-06
start = 2008-10-24 | end = 2018-10-24
start = 2008-12-28 | end = 2018-12-28
start = 2010-03-05 | end = 2020-03-05
start = 2011-12-02 | end = 2021-12-02
start = 2012-11-10 | end = 2022-11-10
start = 2013-01-05 | end = 2023-01-05
start = 2013

In [28]:
# sinal 1
for data_alta_vol in df[df['evento_1'] == True].index:
    print("start =", str(Timestamp(data_alta_vol) - DateOffset(years=10))[:10], "| end =", str(data_alta_vol)[:10])

start = 1998-10-13 | end = 2008-10-13
start = 2010-03-12 | end = 2020-03-12
